In [40]:
import socket
socket.setdefaulttimeout(600)
import time
from typing import Optional, Any
import googleapiclient.errors
from googleapiclient.discovery import Resource
import pandas as pd
import datetime
import sqlite3
# import pymysql
import pandas.io.sql as psql
from datetime import datetime as dt
import numpy as np
import pandas.tseries.offsets as offsets
# import sqlalchemy as sqa

# import matplotlib.pyplot as plt
import python_ss.python_ss as ps
import gspread
# from oauth2client.service_account import ServiceAccountCredentials
import json
import gspread

import os
import ast
import db_dtypes
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import secretmanager

In [41]:
def get_ss(spreadsheet_id: str, range_name: str, service: Resource) -> pd.DataFrame:
    """
    Googleスプレッドシートからデータを取得し、pandas DataFrameとして返します。
    
    特徴:
    - タイムアウト対策済み (socket設定)
    - `fields='values'`による通信量の削減と高速化
    - 1行目をヘッダーとして自動認識（KeyError対策）
    - ネットワークエラー時の自動リトライ機能
    
    Args:
        spreadsheet_id (str): スプレッドシートID
        range_name (str): 範囲指定（例: 'Sheet1!A:M'）
        service (Resource): Google Sheets APIのサービスオブジェクト
        
    Returns:
        pd.DataFrame: 取得したデータ（1行目をカラム名として設定）
    """
    max_retries = 3
    retry_delay = 5  # 初期待機時間（秒）
    
    for attempt in range(max_retries):
        try:
            # スプレッドシートAPIのリクエスト構築
            # 【高速化ポイント】 fields="values" を指定
            # これにより、APIはメタデータを含めず「値のみ」を返すため、
            # レスポンスサイズが小さくなり、通信と解析が高速化します。
            result = service.spreadsheets().values().get(
                spreadsheetId=spreadsheet_id,
                range=range_name,
                fields="values"
            ).execute()
            
            values = result.get('values', [])
            
            # データが存在しない場合の処理
            if not values:
                print(f"[{range_name}] データが取得できませんでした（空です）。")
                return pd.DataFrame()

            # 【修正ポイント】1行目をヘッダーとして使用
            # これにより "tenki_id" などのカラム名が正しく認識され、
            # 後の pd.merge 等での KeyError を回避できます。
            header = values[0]
            data = values[1:]
            
            # データ行がある場合のみ作成（ヘッダーのみの場合は空DF）
            if data:
                df = pd.DataFrame(data, columns=header)
            else:
                df = pd.DataFrame(columns=header)
                
            return df

        except (socket.timeout, googleapiclient.errors.HttpError) as e:
            print(f"データ取得中にエラーが発生しました ({attempt + 1}/{max_retries})")
            print(f"エラー詳細: {e}")
            
            # 最大回数失敗したらエラーを発生させる
            if attempt == max_retries - 1:
                print("最大リトライ回数に達しました。処理を中断します。")
                raise e
            
            # エクスポネンシャルバックオフ（待機時間を徐々に延ばす）
            wait_time = retry_delay * (2 ** attempt)
            print(f"{wait_time}秒待機して再試行します...")
            time.sleep(wait_time)

    return pd.DataFrame()

In [42]:
def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

In [43]:
# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
access_secret_version('r-group-bigdata', 'CREDENTIALS_SECRET_KEY_WORKER'),
scopes=["https://www.googleapis.com/auth/cloud-platform"],)


z:\Users\suehara\Documents\GitHub\rz\.venv\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [44]:
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', None)

## 重複管理シート_マスタ_更新

In [45]:
#登録重複シート更新用の認証・シート設定
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"Z:\Users\suehara\Documents\python\analysis\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '15qRR-yfJxgCh_TpXwBjBHNAnc8yAeTUCF0-Y_N6GoIo'


## 転機内重複確認

In [46]:
qry = """
SELECT 
  id,
	simei_tyofuku,
	tel_tyofuku
FROM `r-group-bigdata.live_tenki.user_tyofuku`
order by id
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
tktyofuku = client.query(qry).result().to_dataframe()

z:\Users\suehara\Documents\GitHub\rz\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [47]:
tyofuku = tktyofuku

In [48]:
tyofuku

,id,simei_tyofuku,tel_tyofuku
0,1,"4,14,15,16,19,21,22,23,27,34,63,73,81,82,84,90...",NaN
1,4,"1,14,15,16,19,21,22,23,27,34,63,73,81,82,84,90...",NaN
2,5,NaN,NaN
3,8,"18,42,48,65,78,123,4704,7884,7888,7889,8396,89...",NaN
4,12,"71,126914",NaN
...,...,...,...
28476,141147,89723,NaN
28477,141150,94717,NaN
28478,141153,91676,91676
28479,141164,137343,NaN


In [49]:
tyofuku["id"] = tyofuku["id"].astype(str) 

tyofuku.replace([np.inf, -np.inf], np.nan, inplace=True)
tyofuku.fillna('', inplace=True)
tyofuku = tyofuku.values.tolist()

In [50]:
service = ps.get_auth(SCOPES, json_path)
Sheet_NAME_jokyo = '登録重複シート!A'
Sheet_row_jokyo = '2'
RANGE_NAME_jokyo = Sheet_NAME_jokyo+Sheet_row_jokyo
ps.update_ss(SPREADSHEET_ID,RANGE_NAME_jokyo,tyofuku,service)

## 手上げ単価計測の候補者マスタ更新

In [51]:
#2020年1月以降の手上げ情報取得
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '15qRR-yfJxgCh_TpXwBjBHNAnc8yAeTUCF0-Y_N6GoIo'
Sheet_NAME = 'マスタ!A'
Sheet_row = ":B"
RANGE_NAME = Sheet_NAME+Sheet_row
tkinfo = get_ss(SPREADSHEET_ID,RANGE_NAME,service)

In [52]:
data = tkinfo.copy()
data["ID"] = data["ID"].astype(int)
data = data.query('ID >= 46273') #28953

In [53]:
#2019年9月以降の手上げ情報取得
SPREADSHEET_ID = '1fOGhqvCoER3YYv2npDUT1KAB6Fw9fYfQ29Vf52p5Nts'
Sheet_NAME = '候補者状況!A'
Sheet_row = ":S"
RANGE_NAME = Sheet_NAME+Sheet_row
pastteageinfo = get_ss(SPREADSHEET_ID,RANGE_NAME,service)
pastteageinfo["ID"] = pastteageinfo["ID"].astype(int)
pastteageinfo = pastteageinfo.query('ID >= 46273') #28953
pastteageinfo["ID"] = pastteageinfo["ID"].astype(str)
pastteageinfo = pastteageinfo[["ID","登録日時","フリ先","手あげ日付","本手上げ","KN共有日"]]
pastteageinfo

,ID,登録日時,フリ先,手あげ日付,本手上げ,KN共有日
46147,46273,2019/10/01,ロンザン,2019/11/06,NaN,NaN
46148,46274,2019/10/01,NaN,NaN,NaN,NaN
46149,46275,2019/10/01,ロンザン,2020/05/21,NaN,NaN
46150,46276,2019/10/01,送信NG,NaN,NaN,NaN
46151,46277,2019/10/01,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
110918,119995,2024/08/12,ロンザン,2024/08/15,TRUE,2024/08/20
110919,119996,2024/08/12,ロンザン,2024/08/15,TRUE,2024/08/15
110920,119997,2024/08/12,ロンザン,2024/08/15,TRUE,2024/08/15
110921,119998,2024/08/12,NaN,NaN,NaN,NaN


In [54]:
#2020年1月以降の手上げ情報取得
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '15qRR-yfJxgCh_TpXwBjBHNAnc8yAeTUCF0-Y_N6GoIo'
Sheet_NAME = '候補者状況!A'
Sheet_row = ":Y"
RANGE_NAME = Sheet_NAME+Sheet_row
teageinfo = get_ss(SPREADSHEET_ID,RANGE_NAME,service)
teageinfo["ID"] = teageinfo["ID"].astype(int)
teageinfo = teageinfo.query('ID > 74999') #28953
teageinfo["ID"] = teageinfo["ID"].astype(str)
teageinfo = teageinfo[["ID","登録日時","フリ先","手あげ日付","本手上げ","KN共有日"]]
teageinfo

,ID,登録日時,フリ先,手あげ日付,本手上げ,KN共有日
0,120000,2024/08/12,ロンザン,2024/08/15,TRUE,2024/08/21
1,120001,2024/08/12,ロンザン,2024/08/15,TRUE,2024/08/15
2,120002,2024/08/12,ロンザン,2024/08/15,TRUE,2024/08/15
3,120003,2024/08/13,,,,
4,120004,2024/08/13,ロンザン,2024/08/15,TRUE,2024/08/15
...,...,...,...,...,...,...
21045,141169,2026/08/16,ロンザン,2026/08/17,TRUE,2026/08/17
21046,141170,2026/08/16,ロンザン,2026/08/17,TRUE,2026/08/17
21047,141171,2026/08/16,,,,
21048,141172,2026/08/16,,,,


In [55]:
teageinfo = pd.concat([pastteageinfo,teageinfo],axis=0,ignore_index=True)
koshinLIST = teageinfo.copy()
data["ID"] = data["ID"].astype(str)
koshinLIST = pd.merge(data,koshinLIST,how="left",on="ID")
koshinLIST = koshinLIST.drop(columns=["登録日時"])
koshinLIST.replace([np.inf, -np.inf], np.nan, inplace=True)
koshinLIST.fillna('', inplace=True)
koshinLIST = koshinLIST.values.tolist()

In [56]:
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '14vXxjcdlyY463QjrtEnCw7pJ0oLStXZD8Q42IdW2Zd4'
service = ps.get_auth(SCOPES, json_path)
Sheet_NAME = '候補者マスタ!A'
Sheet_row = "184"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,koshinLIST,service)

#1r_ceqW-WvUNSCJs7DvJvme7D0q6H4nx-pVeojpQMCfU

## 未手上げファイル更新

In [57]:
miteage_qry = """
select
	inf.id as tenki_id,
  format_date('%Y/%m/%d',inf.created_at) as torokubi,
	inf.shi as sei,
	inf.mei as mei,
	inf.shi_kana as sei_kana,
	inf.mei_kana as mei_kana,
	"11" as ap_source,
	inf.comp_name as company,
	inf.gyousyu1 as gyoshu,
	inf.syokusyu1 as shokushu,
	inf.layer,
	inf.pref,
	inf.work_pref,
	inf.income,
  EXTRACT(YEAR FROM inf.created_at) - birth_year as nenrei,
	inf.seibetu,
	inf.wish_pref,
	inf.moving,
	concat(inf.birth_year,"/",inf.birth_month,"/",inf.birth_day) as birth_day,
	inf.mob_tel as phon_number,
	inf.email,
    dtl.youyaku
from `r-group-bigdata.live_tenki.user_info` inf
left join `r-group-bigdata.live_tenki.user_detail` dtl on inf.id = dtl.id  
where	replace(inf.shi," ","") != ""
AND	replace(inf.mei," ","") != ""
AND	inf.created_at >= DATE_SUB(CURRENT_DATE(), INTERVAL 1 MONTH)
order by inf.id
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
miteage = client.query(miteage_qry).result().to_dataframe()


z:\Users\suehara\Documents\GitHub\rz\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [58]:
miteageA = miteage

In [59]:
path1 = "//172.16.0.232/CoffeeCrazy3/新規事業室（藤社長）/転機_候補者対応関連/ユーザー情報/未手あげデータ/未手あげ候補者インポート_1.2.xlsx"
path1

'//172.16.0.232/CoffeeCrazy3/新規事業室（藤社長）/転機_候補者対応関連/ユーザー情報/未手あげデータ/未手あげ候補者インポート_1.2.xlsx'

In [60]:
# miteageA = miteageA.applymap(remove_control_characters)

In [61]:
# 制御文字(0x00-0x1f)を除去してExcelに書き込む
# applymapでPython関数を1セルずつ適用するより、正規表現replaceでベクトル化した方が高速
miteageA = miteageA.replace(to_replace=r'[\x00-\x1f]', value='', regex=True)

with pd.ExcelWriter(path1, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    miteageA.to_excel(writer, sheet_name='userinfo', startrow=0, startcol=0, index=False)


## 候補者マスタ情報更新

In [62]:

SPREADSHEET_ID = '14vXxjcdlyY463QjrtEnCw7pJ0oLStXZD8Q42IdW2Zd4'
Sheet_NAME = '候補者マスタ!'
Sheet_RANGE = "A:B"
RANGE_NAME = Sheet_NAME+Sheet_RANGE
tkinfo = get_ss(SPREADSHEET_ID,RANGE_NAME,service)

データ取得中にエラーが発生しました (1/3)
エラー詳細: <HttpError 503 when requesting https://sheets.googleapis.com/v4/spreadsheets/14vXxjcdlyY463QjrtEnCw7pJ0oLStXZD8Q42IdW2Zd4/values/%E5%80%99%E8%A3%9C%E8%80%85%E3%83%9E%E3%82%B9%E3%82%BF%21A%3AB?fields=values&alt=json returned "The service is currently unavailable.". Details: "The service is currently unavailable.">
5秒待機して再試行します...


In [63]:
SPREADSHEET_ID = '15qRR-yfJxgCh_TpXwBjBHNAnc8yAeTUCF0-Y_N6GoIo'
Sheet_NAME = 'マスタ!'
Sheet_RANGE = "A:Y"
RANGE_NAME = f"{Sheet_NAME}{Sheet_RANGE}"

# スプレッドシートからデータ取得
tkmasta = get_ss(SPREADSHEET_ID, RANGE_NAME, service)

# 3. 必要なカラムのみ抽出（メモリ使用量の最小化）
tkmasta = tkmasta[["ID", "age", "layer", "income", "change_job", "comp_name", "syokusyu1", "jokin"]]

In [64]:
tkinfo

,tenki_id,登録日時
0,1070,2016/09/17
1,2748,2017/02/06
2,7024,2017/08/07
3,8539,2017/09/15
4,9502,2017/10/05
...,...,...
86321,141169,2026/08/16
86322,141170,2026/08/16
86323,141171,2026/08/16
86324,141172,2026/08/16


In [65]:
tkinfo = tkinfo.rename(columns={"tenki_id":"ID"})
tkinfo = pd.merge(tkinfo,tkmasta,how="left",on="ID")

In [66]:
import pandas as pd
import numpy as np # pd.isna を使う場合や NaN 処理で必要になる可能性

# --- 1. tkinfo["income"] の変換 ---
print("income カラムの変換を開始...")

# 変換ルールを辞書で定義
income_map = {
    "300万円未満": 200,
    "300～399万円": 300,
    "400～499万円": 400,
    "500～599万円": 500,
    "600～699万円": 600,
    "700～799万円": 700,
    # "800～999万円": 800, # 注意: 下の 800-899, 900-999 と重複。どちらが正しいかご確認ください。リスト通りに含めています。
    "800～899万円": 800,
    "900～999万円": 900,
    "1000～1099万円": 1000,
    "1100～1199万円": 1100,
    "1200～1299万円": 1200,
    "1300～1399万円": 1300,
    "1400～1499万円": 1400,
    "1500～1599万円": 1500,
    "1600～1699万円": 1600,
    "1700～1799万円": 1700,
    "1800～1899万円": 1800,
    "1900～1999万円": 1900,
    "2000万円以上": 2000,
    "2000～2999万円": 2000,
    "3000～3999万円": 3000,
    "4000～4999万円": 4000,
    "5000万円以上": 5000,
}

# .map() を使って値を変換し、元のカラムを上書き
tkinfo["income"] = tkinfo["income"].map(income_map)

# 注意: マッピング辞書にない元の値は NaN になります。
# 必要であれば、NaN を特定の値 (例: 0 や -1) で埋める処理を追加してください。
# 例: tkinfo["income"] = tkinfo["income"].fillna(0)

print("income カラムの変換完了。")


# --- 2. tkinfo["change_job"] の変換 ---
print("change_job カラムの変換を開始...")

# 変換ルールを辞書で定義
change_job_map = {
    "転職経験なし": 0,
    "1回（2社経験）": 1,
    "2回（3社経験）": 2,
    "3回（4社経験）": 3,
    "4回（5社経験）": 4,
    "5回（6社経験）": 5,
    "6回（7社経験）": 6,
    "7回（8社経験）": 7,
    "8回（9社経験）": 8,
    "9回（10社経験）": 9,
    "10回以上（11社以上経験）": 10
}

# .map() を使って値を変換し、元のカラムを上書き
tkinfo["change_job"] = tkinfo["change_job"].map(change_job_map)

# 注意: マッピング辞書にない元の値は NaN になります。
# 必要であれば、NaN を特定の値 (例: -1 など) で埋める処理を追加してください。
# 例: tkinfo["change_job"] = tkinfo["change_job"].fillna(-1)

print("change_job カラムの変換完了。")

# --- 3. tkinfo["jokin"] の変換 ---
print("jokin カラムの変換を開始...")

# 変換ルールを適用する関数を定義 (None, NaN, 数値を安全に処理)
def convert_jokin_status(jokin_value):
    """jokinカラムの値を指定された文字列に変換する"""
    if pd.isna(jokin_value): # None または NaN の場合
        return '特に問わない'
    try:
        jokin_int = int(jokin_value) # 比較のために整数に変換試行
        if jokin_int == 3:
            return '特に問わない'
        elif jokin_int == 0:
            return '常勤'
        elif jokin_int == 1:
            return '非常勤（週2-3日）'
        elif jokin_int == 2:
            return '非常勤（月2-3日' # 指示通りの文字列
        else:
            # 0, 1, 2, 3 以外の数値の場合 (そのまま返すか、エラーとするかなど)
            return jokin_value # ここでは元の値を返す
    except (ValueError, TypeError):
        # 文字列など、整数に変換できない場合 (そのまま返すか、エラーとするかなど)
        return jokin_value # ここでは元の値を返す

# .apply() を使って関数を適用し、元のカラムを上書き
tkinfo['jokin'] = tkinfo['jokin'].apply(convert_jokin_status)

print("jokin カラムの変換完了。")

income カラムの変換を開始...
income カラムの変換完了。
change_job カラムの変換を開始...
change_job カラムの変換完了。
jokin カラムの変換を開始...
jokin カラムの変換完了。


In [67]:
tkinfo = tkinfo[['age', 'layer', 'income', 'change_job', 'comp_name','syokusyu1', 'jokin']]

In [68]:
display(tkinfo.tail(10))

,age,layer,income,change_job,comp_name,syokusyu1,jokin
86316,58,役員クラス,2000.0,0.0,ソニーグループ、ソニーグローバルソリューションズ,経営者・CEO・COO等,常勤
86317,55,部長クラス,2000.0,0.0,みずほ銀行,管理部長,常勤
86318,63,課長クラス,600.0,4.0,アリソンベビー&ペッツジャパン株式会社,営業・企画営業（法人対象）,特に問わない
86319,54,課長クラス,900.0,6.0,株式会社 Genki Global Dining Concepts,国際業務・貿易事務,常勤
86320,69,課長クラス,1000.0,3.0,セコム（株）,人事・労務,特に問わない
86321,59,部長クラス,700.0,5.0,神港テクノス株式会社,経営企画・戦略,常勤
86322,56,部長クラス,1100.0,5.0,㈱フェローテック,総務,特に問わない
86323,49,リーダークラス,400.0,3.0,株式会社住理工九州,生産管理,常勤
86324,59,主任・主査クラス,1000.0,0.0,パナソニック オートモーティブシステムズ株式会社,購買・調達・バイヤー,常勤
86325,59,役員クラス,2000.0,0.0,キリンビール株式会社,営業・企画営業（法人対象）,特に問わない


In [71]:
tkinfo = tkinfo.replace([np.inf, -np.inf], np.nan)
tkinfo = tkinfo.astype(object).fillna('')
tkinfo = tkinfo.values.tolist()

In [72]:
SPREADSHEET_ID = '14vXxjcdlyY463QjrtEnCw7pJ0oLStXZD8Q42IdW2Zd4'
service = ps.get_auth(SCOPES, json_path)
Sheet_NAME = '候補者マスタ!L'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,tkinfo,service)